# Cross-scenario model comparison

Compare the three baseline classifiers and the tuned HistGradientBoosting model across three representative FraudZen scenarios. Each scenario gets its own 60/20/20 train, validation, and test split by SIM, so calls from one subscriber cannot appear in multiple partitions. Thresholds are selected on validation rows only. Test results include classification metrics and a confusion matrix for every model/scenario pair.

The target is the known fraudulent SIM label. Predictors are causal SIM-hour behavioral features from `Op_1_CDRTrace.txt`; the existing feature builder uses current-hour records and prior-hour event counts. The notebook does not use the `International_CDRTrace.txt` add-on, keeping the comparison consistent with the completed pipeline.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix)

PROJECT = Path.cwd()
if PROJECT.name.lower() == 'notebooks':
    PROJECT = PROJECT.parent
DATA_ROOT = PROJECT / 'Fraudzen-dataset' / 'Fraudzen-dataset' / 'FraudZen_Cdrs'
OUT = PROJECT / 'outputs' / 'models' / 'cross_scenario_comparison'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT / 'tools'))
from build_hourly_features import build_features

SEED = 42
SCENARIOS = {
    'scenario_1': DATA_ROOT / 'all_naive' / 'all_naive_12%_200sims',
    'scenario_2': DATA_ROOT / 'all_naive' / 'all_naive_3%_200sims',
    'scenario_3': DATA_ROOT / 'all_naive' / 'all_naive_12%_50sims',
}
missing = [str(p) for p in SCENARIOS.values() if not (p / 'Op_1_CDRTrace.txt').exists() or not (p / 'fraudulentUsers.txt').exists()]
if missing:
    raise FileNotFoundError('Scenario input file(s) missing:\n' + '\n'.join(missing))
print('Project:', PROJECT)
print('Output:', OUT)
print('Scenarios:', list(SCENARIOS))

## Build the same feature table for each scenario

Feature tables are cached under `outputs/model_ready/cross_scenario/`. Set `REBUILD_FEATURES = True` to regenerate them from the supplied raw traces.

In [ ]:
FEATURE_DIR = PROJECT / 'outputs' / 'model_ready' / 'cross_scenario'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
REBUILD_FEATURES = False
frames = {}
feature_summary = []
for name, scenario_path in SCENARIOS.items():
    feature_path = FEATURE_DIR / f'{name}_sim_hour_features.csv'
    if REBUILD_FEATURES or not feature_path.exists():
        total_events, rows = build_features(scenario_path, feature_path, include_international=False)
        print(f'{name}: parsed {total_events:,} events into {rows:,} SIM-hour rows')
    frame = pd.read_csv(feature_path, dtype={'src_number': 'string'}, parse_dates=['hour'])
    frame['src_number'] = frame['src_number'].astype(str)
    frame['fraud'] = frame['fraud'].astype(int)
    frames[name] = frame
    feature_summary.append({'scenario': name, 'rows': len(frame), 'SIMs': frame['src_number'].nunique(), 'fraud_SIMs': frame.groupby('src_number')['fraud'].max().sum(), 'hour_start': frame['hour'].min(), 'hour_end': frame['hour'].max()})
pd.DataFrame(feature_summary)

## Train and evaluate four models

Models: Logistic Regression, Random Forest, baseline HistGradientBoosting, and tuned HistGradientBoosting. The tuned settings are fixed from the earlier primary-scenario tuning stage; they are not retuned on the held-out test partition. The primary metric is PR-AUC because fraud is the minority class.

In [ ]:
def make_models():
    return {
        'Logistic Regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=3000, random_state=SEED)),
        'Random Forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)),
        'HistGradientBoosting': make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_iter=250, learning_rate=.06, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)),
        'Tuned HistGradientBoosting': make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_iter=150, learning_rate=.04, max_leaf_nodes=7, l2_regularization=1.0, random_state=SEED)),
    }

def grouped_partitions(X, y, groups):
    first = GroupShuffleSplit(n_splits=1, test_size=.40, random_state=SEED)
    train_idx, remaining = next(first.split(X, y, groups))
    second = GroupShuffleSplit(n_splits=1, test_size=.50, random_state=SEED + 1)
    val_rel, test_rel = next(second.split(X.iloc[remaining], y.iloc[remaining], groups.iloc[remaining]))
    return train_idx, remaining[val_rel], remaining[test_rel]

def choose_threshold(actual, probability):
    thresholds = np.linspace(.01, .99, 99)
    scores = [f1_score(actual, probability >= t, zero_division=0) for t in thresholds]
    best = int(np.argmax(scores))
    return float(thresholds[best]), float(scores[best])

all_metrics, all_matrices, all_predictions = [], [], []
for scenario, frame in frames.items():
    excluded = {'fraud', 'src_number', 'hour', 'known_fraud', 'record_id', 'event_key'}
    columns = [c for c in frame.select_dtypes(include='number').columns if c not in excluded]
    X = frame[columns].replace([np.inf, -np.inf], np.nan)
    y = frame['fraud']
    groups = frame['src_number'].astype(str)
    train_idx, val_idx, test_idx = grouped_partitions(X, y, groups)
    group_sets = [set(groups.iloc[idx]) for idx in (train_idx, val_idx, test_idx)]
    assert not (group_sets[0] & group_sets[1] or group_sets[0] & group_sets[2] or group_sets[1] & group_sets[2]), 'SIM leakage across partitions'
    if any(y.iloc[idx].nunique() < 2 for idx in (train_idx, val_idx, test_idx)):
        raise ValueError(f'{scenario}: a split lacks one class; adjust the split seed/strategy')
    print(f'\n{scenario}: features={len(columns)}, SIMs train/val/test={[len(s) for s in group_sets]}, fraud SIMs={[int(y.iloc[idx].groupby(groups.iloc[idx]).max().sum()) for idx in (train_idx, val_idx, test_idx)]}')
    for model_name, model in make_models().items():
        if model_name in {'HistGradientBoosting', 'Tuned HistGradientBoosting'}:
            weights = compute_sample_weight(class_weight='balanced', y=y.iloc[train_idx])
            model.fit(X.iloc[train_idx], y.iloc[train_idx], histgradientboostingclassifier__sample_weight=weights)
        else:
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
        val_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        threshold, val_f1 = choose_threshold(y.iloc[val_idx].to_numpy(), val_prob)
        test_prob = model.predict_proba(X.iloc[test_idx])[:, 1]
        actual = y.iloc[test_idx].to_numpy()
        predicted = (test_prob >= threshold).astype(int)
        cm = confusion_matrix(actual, predicted, labels=[0, 1])
        row = {'scenario': scenario, 'model': model_name, 'threshold': threshold, 'validation_f1': val_f1, 'test_accuracy': accuracy_score(actual, predicted), 'test_balanced_accuracy': balanced_accuracy_score(actual, predicted), 'test_precision': precision_score(actual, predicted, zero_division=0), 'test_recall': recall_score(actual, predicted, zero_division=0), 'test_f1': f1_score(actual, predicted, zero_division=0), 'test_pr_auc': average_precision_score(actual, test_prob), 'test_roc_auc': roc_auc_score(actual, test_prob), 'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1], 'test_rows': len(test_idx), 'train_sims': len(group_sets[0]), 'validation_sims': len(group_sets[1]), 'test_sims': len(group_sets[2]), 'features': len(columns)}
        all_metrics.append(row)
        all_matrices.append((scenario, model_name, cm))
        all_predictions.extend({'scenario': scenario, 'model': model_name, 'sim': groups.iloc[test_idx].to_numpy()[i], 'hour': frame.iloc[test_idx]['hour'].astype(str).to_numpy()[i], 'actual': int(actual[i]), 'probability': float(test_prob[i]), 'threshold': threshold, 'predicted': int(predicted[i])} for i in range(len(test_idx)))
        print(f"  {model_name}: PR-AUC={row['test_pr_auc']:.4f}, F1={row['test_f1']:.4f}, recall={row['test_recall']:.4f}, threshold={threshold:.2f}")

metrics = pd.DataFrame(all_metrics).sort_values(['scenario', 'test_pr_auc'], ascending=[True, False]).reset_index(drop=True)
predictions = pd.DataFrame(all_predictions)
metrics.to_csv(OUT / 'cross_scenario_metrics.csv', index=False)
predictions.to_csv(OUT / 'cross_scenario_test_predictions.csv', index=False)
metrics

## Confusion matrices and comparison plot

Each matrix uses its model's validation-selected threshold. Compare PR-AUC and recall alongside false positives and false negatives; accuracy alone can hide weak minority-class detection.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
for scenario, model_name, cm in all_matrices:
    slug = f"{scenario}_{model_name.lower().replace(' ', '_')}"
    pd.DataFrame(cm, index=['actual_legitimate', 'actual_fraudulent'], columns=['predicted_legitimate', 'predicted_fraudulent']).to_csv(OUT / f'confusion_matrix_{slug}.csv')
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraudulent']).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'{scenario}\n{model_name}')
    fig.tight_layout()
    fig.savefig(OUT / f'confusion_matrix_{slug}.png', dpi=150)
    plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
metrics.pivot(index='model', columns='scenario', values='test_pr_auc').plot(kind='bar', ax=ax)
ax.set_ylabel('Held-out test PR-AUC')
ax.set_xlabel('Model')
ax.set_title('Four-model comparison across selected FraudZen scenarios')
ax.legend(title='Scenario', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(OUT / 'cross_scenario_test_pr_auc.png', dpi=160, bbox_inches='tight')
plt.show()
print('Metrics, predictions, confusion matrices, and chart saved to:', OUT)